# Reasoning prompt techniques

A small Groq lab for two techniques: **chain-of-thought prompting** and **ReAct**. Both use the same syllabus task, but they solve different problems.

## Setup

The notebook loads `GROQ_API_KEY` from the project-root `.env` file.

In [ ]:
%pip install -q openai python-dotenv

import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ENV_FILE = PROJECT_ROOT / '.env'
load_dotenv(ENV_FILE)
if not os.getenv('GROQ_API_KEY'):
    raise RuntimeError(f'GROQ_API_KEY was not found in {ENV_FILE}')

MODEL = os.getenv('GROQ_MODEL', 'openai/gpt-oss-20b')
client = OpenAI(
    api_key=os.environ['GROQ_API_KEY'],
    base_url='https://api.groq.com/openai/v1',
)

def ask(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.2,
    )
    return response.choices[0].message.content

print(f'Using Groq model: {MODEL}')

In [ ]:
SYLLABUS = '''
Course: Cloud-Native Application Development

Students will design and test a REST API with authentication.
Students will deploy a containerised service using a continuous delivery pipeline.
Students will gain exposure to cloud-native systems.
'''

## 1. Chain-of-thought prompting: work through a rule before deciding

Use this when the model has several rules to apply. Ask for a **short, reviewable checklist**, rather than treating an unverified explanation as proof.

In [ ]:
cot_prompt = f'''
Is this syllabus statement an assessable course outcome?

Statement: Students will gain exposure to cloud-native systems.

Work through this short checklist before deciding:
1. Quote the action verb, if one exists.
2. Decide whether that action is observable and assessable.
3. Return the final decision.

Return exactly:
CHECK: <one concise sentence>
DECISION: ASSESSABLE OUTCOME or NEEDS REVIEW

Use this syllabus only as context:
{SYLLABUS}
'''

print(ask(cot_prompt))

## 2. ReAct: reason → act → observe → answer

Use this when a question needs source-grounded or current information. The application performs a controlled action, then gives the model the observation. The model is not allowed to invent evidence outside that observation.

In [ ]:
def find_syllabus_evidence(term):
    """A read-only action: retrieve matching syllabus lines."""
    return [line for line in SYLLABUS.splitlines() if term.lower() in line.lower()]

question = 'Does this syllabus explicitly say that students will deploy a service?'
action = "find_syllabus_evidence('deploy')"
observation = find_syllabus_evidence('deploy')

print('QUESTION:', question)
print('ACTION:', action)
print('OBSERVATION:', observation)

react_prompt = f'''
Answer the question using only the observation returned by the approved lookup.
If the observation is empty, say that the syllabus does not provide enough evidence.

QUESTION: {question}
OBSERVATION: {observation}

Return exactly:
ANSWER: <yes, no, or insufficient evidence>
EVIDENCE: <quote from the observation, or 'none'>
'''

print(ask(react_prompt))

## Try it

- Change the statement in the chain-of-thought example. Does the checklist still identify the right boundary?
- Change the ReAct lookup term to one that does not occur. Does the answer correctly return insufficient evidence?

Use chain-of-thought to clarify a multi-step decision. Use ReAct when the decision requires an observation from an approved tool or data source.